# Notebook 02 — Labels and Text for the Safety Baseline

**Author:** Arbaz Khalid — Safety & Governance Module  
**Status:** Build the labelled 311 dataset (3 classes) for the first baseline

## Purpose
1. Download the full Ida-window 311 data (63,709 reports).
2. Apply draft weak labels: flood, road_blockage, non_hazard.
3. Build a short text from each report's fields (311 has no citizen free text).
4. Save the result for the baseline model.

## Known limitations
- power_outage is dropped: 311 `ELECTRIC / POWER OUTAGE` reports are all HPD residential-building complaints, not grid failures.
- Labels come from category fields, and the text is built from those same fields, so scores on this data show the pipeline works, not real-world performance.

In [1]:
import os
import re
import requests
import pandas as pd

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
WHERE = "created_date between '2021-08-28T00:00:00' and '2021-09-03T00:00:00'"
COLUMNS = ("unique_key, created_date, agency, complaint_type, descriptor, location_type, "
           "incident_address, street_name, cross_street_1, cross_street_2, borough, latitude, longitude")

def soda_get(params):
    r = requests.get(URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def pull_all(where, page=10000, max_rows=200000):
    rows, offset = [], 0
    while offset < max_rows:
        batch = soda_get({"$select": COLUMNS, "$where": where, "$order": "unique_key",
                          "$limit": page, "$offset": offset})
        if not batch:
            break
        rows.extend(batch)
        offset += page
    return pd.DataFrame(rows)

In [2]:
df = pull_all(WHERE)
df["created_date"] = pd.to_datetime(df["created_date"])
print(df.shape)
assert df["unique_key"].is_unique, "duplicate rows - paging problem"

os.makedirs("data/raw", exist_ok=True)
df.to_csv("data/raw/nyc311_ida_full.csv", index=False)
print("Saved data/raw/nyc311_ida_full.csv")

(63709, 13)
Saved data/raw/nyc311_ida_full.csv


In [3]:
RULES = [
    ("Sewer", "Sewer Backup", "flood"),
    ("Sewer", "Street Flooding", "flood"),
    ("Sewer", "Catch Basin", "flood"),
    ("Sewer", "Manhole Overflow", "flood"),
    ("Water System", "Excessive Water In Basement", "flood"),
    ("Damaged Tree", "Entire Tree Has Fallen", "road_blockage"),
    ("Damaged Tree", "Branch or Limb Has Fallen", "road_blockage"),
    ("Damaged Tree", "Tree Leaning/Uprooted", "road_blockage"),
    ("Street Condition", "Cave-in", "road_blockage"),
]

def weak_label(ctype, descriptor):
    for rule_type, keyword, label in RULES:
        if ctype == rule_type and keyword.lower() in str(descriptor).lower():
            return label
    return "non_hazard"

df["label"] = [weak_label(c, d) for c, d in zip(df["complaint_type"], df["descriptor"])]
df["label"].value_counts()

label
non_hazard       57351
flood             5142
road_blockage     1216
Name: count, dtype: int64

In [4]:
def clean_descriptor(d):
    d = re.sub(r"\(.*?\)", "", str(d))        # remove codes like (SA) or (Use Comments)
    return re.sub(r"\s+", " ", d).strip()

def build_text(row):
    place = row["incident_address"] if pd.notna(row["incident_address"]) else row["street_name"]
    place = "unknown location" if pd.isna(place) else re.sub(r"\s+", " ", str(place)).strip().title()
    borough = "" if pd.isna(row["borough"]) else str(row["borough"]).title()
    return f"{row['complaint_type']}: {clean_descriptor(row['descriptor'])} at {place}, {borough}"

df["text"] = df.apply(build_text, axis=1)
df[["text", "label"]].sample(8, random_state=1)

,text,label
60779,"Sewer: Sewer Backup at 157-38 25 Avenue, Queens",flood
665,Noise - Residential: Loud Music/Party at 38 Lo...,non_hazard
54147,Illegal Parking: Double Parked Blocking Vehicl...,non_hazard
30444,Derelict Vehicles: Derelict Vehicles at 533 We...,non_hazard
44794,Illegal Parking: Posted Parking Sign Violation...,non_hazard
56981,Elevator: Elevator - Multiple Devices On Prope...,non_hazard
9978,"Water System: Dirty Water at 23-24 96 Street, ...",non_hazard
39732,Noise - Street/Sidewalk: Loud Music/Party at 1...,non_hazard


In [5]:
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/nyc311_ida_labeled.csv", index=False)
print("Saved", len(df), "rows")

Saved 63709 rows
